# Example queries: `calculated_column` (resstock_oedi)

Auto-generated from `tests/query_snapshots/calculated_column.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object


In [ ]:
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
)


## `calculated_column_elec_minus_gas`

Annual aggregation using a get_calculated_column-built enduse: electricity total minus natural gas total. Pins the operator-precedence and column-resolution path through get_calculated_column. The column_expr placeholders ($ELECTRICITY_TOTAL, $NATURAL_GAS_TOTAL) get resolved to per-schema column names by the specialized test function before construction.


In [ ]:
result = bsq.query(
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x11a3d1c40; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  elec_minus_gas
                  Mobile Home           391 9.864994e+04   -4.898078e+08
Multi-Family with 2 - 4 Units           469 1.183295e+05   -1.999671e+08
   Multi-Family with 5+ Units          2001 5.048556e+05    7.832083e+08
       Single-Family Attached           664 1.675283e+05   -5.635194e+08
       Single-Family Detached          5900 1.488580e+06   -1.489333e+10


## `calculated_column_ts_pivot_upgrade_pair`

TS calculated column on the pivot path: table='timeseries' + annual_only=False + upgrade_id != 0 + applied_only=False forces the upgrade-pair pivot subquery (NOT the upgrade_only short-circuit). The Label produced by get_calculated_column wraps an arithmetic expression of raw ts columns; the pivot must wrap that whole expression in a per-side CASE, not look up Label.name on ts.c. Without this entry, the pivot path's calculated-column handling is uncovered.


In [ ]:
result = bsq.query(
    annual_only=False,
    upgrade_id='1',
    applied_only=False,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x11a57d6d0; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (12, 6)
state  timestamp  sample_count  units_count rows_per_sample  elec_minus_gas
   CO 2018-01-01          9425 2.377943e+06            2976    4.093880e+09
   CO 2018-02-01          9425 2.377943e+06            2688    4.134305e+09
   CO 2018-03-01          9425 2.377943e+06            2976    1.905547e+09
   CO 2018-04-01          9425 2.377943e+06            2880    1.421178e+09
   CO 2018-05-01          9425 2.377943e+06            2976    9.261339e+08


## `calculated_column_ts_pivot_with_baseline`

TS calculated column on the pivot path with include_baseline=True + applied_only=True. This combination forces the pivot subquery (upgrade_only short-circuit only fires when both include_baseline and include_savings are False) AND exercises the applied_in synthesis path that filters to applicable buildings. Together with the upgrade_pair entry above, these two pin both branches of the TS pivot's calculated-column handling.


In [ ]:
result = bsq.query(
    annual_only=False,
    upgrade_id='1',
    applied_only=True,
    include_baseline=True,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
    enduses=[<sqlalchemy.sql.elements.Label at 0x11a622150; elec_minus_gas>],
)
result.head() if hasattr(result, 'head') else result


# shape: (12, 7)
state  timestamp  sample_count  units_count rows_per_sample  elec_minus_gas__baseline  elec_minus_gas__upgrade
   CO 2018-01-01          9071 2.288628e+06            2976             -4.524833e+09             4.052893e+09
   CO 2018-02-01          9071 2.288628e+06            2688             -4.221866e+09             4.097495e+09
   CO 2018-03-01          9071 2.288628e+06            2976             -2.057556e+09             1.864938e+09
   CO 2018-04-01          9071 2.288628e+06            2880             -1.211228e+09             1.382829e+09
   CO 2018-05-01          9071 2.288628e+06            2976              2.778346e+08             8.845872e+08
